## 1. DESCARGAR DATOS

In [1]:
from pathlib import Path
import openpyxl
import yfinance as yf

workspace_root = Path.cwd().resolve().parents[1]
outfile = workspace_root / "MSFT_historical.csv"

history = yf.download("MSFT", period="max", auto_adjust=False, progress=False)
if history.empty:
    raise RuntimeError("No se pudieron descargar datos históricos para MSFT")

history.to_csv(outfile)
print(f"CSV creado: {outfile}")
print(f"Filas descargadas: {len(history):,}")

CSV creado: /home/hacker-m/Desktop/UDD/2026/Trimestre_01/AI_Workshop/Group/iit414w-lab01-TheUltrakills/MSFT_historical.csv
Filas descargadas: 10,122


In [2]:
from pathlib import Path

import pandas as pd
import yfinance as yf

workspace_root = Path.cwd().resolve()
csv_candidates = [
    workspace_root / "MSFT_historical.csv",
    workspace_root / "EvalProyectos" / "Tarea 2" / "MSFT_historical.csv",
]
msft_csv = next((path for path in csv_candidates if path.exists()), None)
if msft_csv is None:
    raise FileNotFoundError("No se encontró MSFT_historical.csv ni en la raíz del workspace ni en la carpeta de la tarea")

msft_raw = pd.read_csv(msft_csv, header=[0, 1], index_col=0, parse_dates=True)
msft_prices = msft_raw["Adj Close"].squeeze("columns")

spy_history = yf.download(
    "SPY",
    start=msft_prices.index.min(),
    end=msft_prices.index.max() + pd.Timedelta(days=1),
    auto_adjust=False,
    progress=False,
)
if spy_history.empty:
    raise RuntimeError("No se pudieron descargar datos de mercado para SPY")

spy_prices = spy_history["Adj Close"].squeeze("columns")



### Respuesta 1

 El beta calculado es de 1.0978 Beta 5YL Daily

 La frecuencia de los datos fue diaria

 El largo de la muestra es de 8380 observaciones de rendimiento

 Esta empresa fue elegida debido a su gran tamaño y flujo en el mercado, lo que da por hecho que tendrán una gran cantidad de data historica para usar

Diferencia con YahooFin = 0.0078 Beta 5YL Monthly



ej3 : Tasa

In [3]:
rf_history = yf.download(
    "^IRX",
    start=msft_prices.index.min(),
    end=msft_prices.index.max() + pd.Timedelta(days=1),
    auto_adjust=False,
    progress=False,
)

if rf_history.empty:
    raise RuntimeError("No se pudieron descargar datos de ^IRX")

rf_df = rf_history[["Adj Close"]].rename(columns={"Adj Close": "RF_anual"})
rf_df["RF_diaria"] = (1 + rf_df["RF_anual"] / 100) ** (1 / 252) - 1

Todos parten en la misma fecha:

In [4]:
spy_history

Price,Adj Close,Close,High,Low,Open,Volume
Ticker,SPY,SPY,SPY,SPY,SPY,SPY
Date,,,,,,
1993-01-29,24.175381,43.937500,43.968750,43.750000,43.968750,1003200
1993-02-01,24.347332,44.250000,44.250000,43.968750,43.968750,480500
1993-02-02,24.398909,44.343750,44.375000,44.125000,44.218750,201300
1993-02-03,24.656826,44.812500,44.843750,44.375000,44.406250,529400
1993-02-04,24.759983,45.000000,45.093750,44.468750,44.968750,531500
...,...,...,...,...,...,...
2026-05-11,739.299988,739.299988,740.789978,736.450012,736.450012,44024000
2026-05-12,738.179993,738.179993,738.840027,731.830017,736.890015,54185300


In [5]:
# Align all dataframes to the same common date range and recompute returns/beta
common_start = max(msft_prices.index.min(), spy_prices.index.min(), rf_df.index.min())
common_end = min(msft_prices.index.max(), spy_prices.index.max(), rf_df.index.max())

msft_raw = msft_raw.loc[common_start:common_end]
msft_prices = msft_raw["Adj Close"].squeeze("columns")

spy_history = spy_history.loc[common_start:common_end]
spy_prices = spy_history["Adj Close"].squeeze("columns")

rf_df = rf_df.loc[common_start:common_end]




In [6]:
# Extra alignment step: force all datasets to share the same dates
# Build a strict common index from MSFT and SPY trading days
common_index = msft_prices.index.intersection(spy_prices.index).sort_values()

# Keep MSFT and SPY on that exact index
msft_raw = msft_raw.loc[common_index]
spy_history = spy_history.loc[common_index]

msft_prices = msft_raw["Adj Close"].squeeze("columns")
spy_prices = spy_history["Adj Close"].squeeze("columns")

# Reindex RF and fill missing ^IRX dates with latest available rate
rf_df = rf_df.reindex(common_index).ffill().bfill()

# Recompute returns and align RF to returns index
returns = pd.concat(
    [
        msft_prices.pct_change().rename("MSFT"),
        spy_prices.pct_change().rename("SPY"),
    ],
    axis=1,
).dropna()
rf_df = rf_df.reindex(returns.index).ffill().bfill()

# Recompute beta on fully aligned data
sigma_im = returns["MSFT"].cov(returns["SPY"])
sigma_m2 = returns["SPY"].var()
beta_i = sigma_im / sigma_m2

print(f"Fechas comunes MSFT/SPY: {len(common_index):,}")
print(f"Fechas finales alineadas MSFT/SPY/RF: {len(returns.index):,}")
print(f"Rango final: {returns.index.min().date()} a {returns.index.max().date()}")
print(f"beta_i (recalculado) = {beta_i:.6f}")

Fechas comunes MSFT/SPY: 8,381
Fechas finales alineadas MSFT/SPY/RF: 8,380
Rango final: 1993-02-01 a 2026-05-15
beta_i (recalculado) = 1.097801


## EXPORTADO DE DATOS A EXCEL

In [7]:
rf_daily = rf_df["RF_diaria"].rename("RF_daily")

combined_df = pd.concat(
    [
        msft_prices.pct_change().rename("MSFT"),
        spy_prices.pct_change().rename("SPY"),
        rf_daily,
    ],
    axis=1,
    join="inner",
).dropna()

combined_df.head()

,MSFT,SPY,RF_daily
Date,,,
1993-02-01,0.011561,0.007113,0.000113
1993-02-02,0.017143,0.002118,0.000116
1993-02-03,-0.007023,0.010571,0.000115
1993-02-04,-0.038189,0.004184,0.000113
1993-02-05,0.047059,-0.000695,0.000113


In [8]:

# Export MSFT and SPY data to Excel with multiple sheets
excel_file = workspace_root / "MSFT_SPY_data.xlsx"


with pd.ExcelWriter(excel_file, engine="openpyxl") as writer:
    msft_raw.to_excel(writer, sheet_name="MSFT")
    spy_history.to_excel(writer, sheet_name="SPY")
    returns.to_excel(writer, sheet_name="Retornos")
    combined_df.to_excel(writer, sheet_name="RF")


print(f"Archivo Excel creado: {excel_file}")
print(f"Hojas: 'MSFT', 'SPY', 'Retornos' y '^IRX' ")


Archivo Excel creado: /home/hacker-m/Desktop/UDD/2026/Trimestre_01/AI_Workshop/Group/iit414w-lab01-TheUltrakills/EvalProyectos/Tarea 2/MSFT_SPY_data.xlsx
Hojas: 'MSFT', 'SPY', 'Retornos' y '^IRX' 


## DATOS EJ 8

In [9]:
import yfinance as yf

ticker = yf.Ticker("MSFT")
info = ticker.info

market_cap = info['marketCap']  # Valor de mercado del equity
shares_outstanding = info['sharesOutstanding']  # Acciones en circulación
current_price = info['currentPrice']  # Precio actual

print(f"Market Cap: ${market_cap:,}")
print(f"Shares Outstanding: {shares_outstanding:,}")
print(f"Current Price: ${current_price}")

Market Cap: $3,134,205,460,480
Shares Outstanding: 7,428,434,704
Current Price: $421.92


In [19]:
financials

,2025-06-30,2024-06-30,2023-06-30,2022-06-30
Tax Effect Of Unusual Items,-7.708800e+07,-9.991800e+07,-2.850000e+06,4.375400e+07
Tax Rate For Calcs,1.760000e-01,1.820000e-01,1.900000e-01,1.310000e-01
Normalized EBITDA,1.606030e+11,1.335580e+11,1.051550e+11,9.990500e+10
Total Unusual Items,-4.380000e+08,-5.490000e+08,-1.500000e+07,3.340000e+08
Total Unusual Items Excluding Goodwill,-4.380000e+08,-5.490000e+08,-1.500000e+07,3.340000e+08
Net Income From Continuing Operation Net Minority Interest,1.018320e+11,8.813600e+10,7.236100e+10,7.273800e+10
Reconciled Depreciation,3.415300e+10,2.228700e+10,1.386100e+10,1.446000e+10
Reconciled Cost Of Revenue,8.783100e+10,7.411400e+10,6.586300e+10,6.265000e+10
EBITDA,1.601650e+11,1.330090e+11,1.051400e+11,1.002390e+11
EBIT,1.260120e+11,1.107220e+11,9.127900e+10,8.577900e+10


In [27]:
# Income statement
financials = ticker.financials  # O ticker.financials para anual
financials_2024 = financials["2024-06-30"]

pretax_2024 = financials_2024["Pretax Income"]
income_tax_2024 = financials_2024["Tax Provision"]

# Tasa impositiva = Income Tax / Pretax Income
print(f"Pretax 2024 {pretax_2024} | Income Tax 2024 {income_tax_2024}")
tasa_impositiva = income_tax_2024 / pretax_2024
tasa_impositiva

Pretax 2024 107787000000.0 | Income Tax 2024 19651000000.0


np.float64(0.18231326597827197)

In [28]:
import yfinance as yf

# Obtener datos de MSFT
ticker = yf.Ticker("MSFT")
info = ticker.info

market_cap = info['marketCap']
current_price = info['currentPrice']

# Datos que tienes
beta_l = 1.0978  # Beta apalancada
debt_2024 = 33_315_000  # USD
tax_rate = tasa_impositiva  # 15% (tasa impositiva corporativa)

# Calcular beta desapalancada
D_E_ratio = debt_2024 / market_cap
beta_u = beta_l / (1 + (1 - tax_rate) * D_E_ratio)

print(f"Market Cap MSFT: ${market_cap:,.0f}")
print(f"Deuda 2024: ${debt_2024:,.0f}")
print(f"D/E Ratio: {D_E_ratio:.6f}")
print(f"Beta Apalancada: {beta_l:.6f}")
print(f"Beta Desapalancada: {beta_u:.6f}")

Market Cap MSFT: $3,134,205,460,480
Deuda 2024: $33,315,000
D/E Ratio: 0.000011
Beta Apalancada: 1.097800
Beta Desapalancada: 1.097790
